#### Enigma Part
* basically we need it to encrypt our cribs
* which means that it encrypts the cribs and then checks back with the bombe
* keeps doing this so that it can find whats right and whats wrong
* without this part its like having a key but no lock to put it in

In [5]:
import time
import string
from typing import List, Tuple, Optional, Dict
from itertools import product
from datetime import datetime
import os

class EnigmaMachine:
    """3-rotor Enigma machine implementation"""
    
    # Historical rotor wirings (Enigma I)
    ROTORS = {
        'I':   'EKMFLGDQVZNTOWYHXUSPAIBRCJ',
        'II':  'AJDKSIRUXBLHWTMCQGZNPYFVOE',
        'III': 'BDFHJLCPRTXVZNYEIWGAKMUSQO',
        'IV':  'ESOVPZJAYQUIRHXLNFTGKDCMWB',
        'V':   'VZBRGITYUPSDNHLXAWMJQOFECK'
    }
    
    # Notch positions (when rotor turns the next one)
    NOTCHES = {
        'I': 'Q',
        'II': 'E',
        'III': 'V',
        'IV': 'J',
        'V': 'Z'
    }
    
    # Reflector B (most common)
    REFLECTOR_B = 'YRUHQSLDPXNGOKMIEBFZCWVJAT'
    ALPHABET = string.ascii_uppercase
    
    def __init__(self, rotors: Tuple[str, str, str], 
                 positions: Tuple[int, int, int],
                 ring_settings: Tuple[int, int, int] = (0, 0, 0),
                 plugboard: Dict[str, str] = None):
        """
        Initialize Enigma machine
        
        Args:
            rotors: Tuple of 3 rotor names (e.g., ('I', 'II', 'III'))
            positions: Initial rotor positions (0-25)
            ring_settings: Ring settings for each rotor (0-25)
            plugboard: Dictionary of plugboard connections
        """
        self.rotors = [self.ROTORS[r] for r in rotors]
        self.rotor_names = rotors
        self.positions = list(positions)
        self.ring_settings = list(ring_settings)
        self.notches = [self.NOTCHES[r] for r in rotors]
        self.reflector = self.REFLECTOR_B
        self.plugboard = plugboard or {}
        
    def _plugboard_swap(self, char: str) -> str:
        """Apply plugboard transformation"""
        return self.plugboard.get(char, char)
    
    def _rotate_rotors(self):
        """Rotate rotors according to Enigma mechanism (including double-stepping)"""
        # Check if middle rotor is at notch (causes double-stepping)
        if self.ALPHABET[self.positions[1]] == self.notches[1]:
            self.positions[1] = (self.positions[1] + 1) % 26
            self.positions[2] = (self.positions[2] + 1) % 26
        # Check if right rotor is at notch
        elif self.ALPHABET[self.positions[0]] == self.notches[0]:
            self.positions[1] = (self.positions[1] + 1) % 26
        
        # Always rotate rightmost rotor
        self.positions[0] = (self.positions[0] + 1) % 26
    
    def _encode_through_rotor(self, char_index: int, rotor_index: int, 
                             forward: bool = True) -> int:
        """Pass character through a rotor"""
        rotor = self.rotors[rotor_index]
        position = self.positions[rotor_index]
        ring = self.ring_settings[rotor_index]
        
        if forward:
            shifted = (char_index + position - ring) % 26
            encoded = self.ALPHABET.index(rotor[shifted])
            return (encoded - position + ring) % 26
        else:
            shifted = (char_index + position - ring) % 26
            encoded = rotor.index(self.ALPHABET[shifted])
            return (encoded - position + ring) % 26
    
    def _encode_through_reflector(self, char_index: int) -> int:
        """Pass character through reflector"""
        return self.ALPHABET.index(self.reflector[char_index])
    
    def encrypt_char(self, char: str) -> str:
        """Encrypt a single character"""
        if char not in self.ALPHABET:
            return char
        
        # Rotate rotors BEFORE encryption
        self._rotate_rotors()
        
        # Apply plugboard
        char = self._plugboard_swap(char)
        char_index = self.ALPHABET.index(char)
        
        # Pass through rotors (right to left)
        for i in range(3):
            char_index = self._encode_through_rotor(char_index, i, forward=True)
        
        # Pass through reflector
        char_index = self._encode_through_reflector(char_index)
        
        # Pass back through rotors (left to right)
        for i in range(2, -1, -1):
            char_index = self._encode_through_rotor(char_index, i, forward=False)
        
        # Apply plugboard again
        result = self.ALPHABET[char_index]
        result = self._plugboard_swap(result)
        
        return result
    
    def encrypt(self, text: str) -> str:
        """Encrypt a message"""
        return ''.join(self.encrypt_char(c.upper()) for c in text if c.isalpha())



#### Bombe Decoding Part

In [6]:
class Bombe:
    """
    Digital version of Turing's Bombe
    
    Exploits the fact that in Enigma, no letter encrypts to itself.
    Uses known plaintext fragments (cribs) to test possible rotor settings.
    Ex: "Heil Hitler" was commonly used at the end of German messages in WW2,
    making it much faster to find the original settings.
    """
    
    def __init__(self, rotors_to_test: List[Tuple[str, str, str]] = None):
        """
        Initialize Bombe
        
        Args:
            rotors_to_test: List of rotor combinations to test
                          If None, tests all permutations of rotors I, II, III
        """
        if rotors_to_test is None:
            # Test all permutations of rotors I, II, III
            self.rotors_to_test = [
                ('I', 'II', 'III'),
                ('I', 'III', 'II'),
                ('II', 'I', 'III'),
                ('II', 'III', 'I'),
                ('III', 'I', 'II'),
                ('III', 'II', 'I')
            ]
        else:
            self.rotors_to_test = rotors_to_test
    
    def _test_crib(self, ciphertext: str, crib: str, 
                   rotors: Tuple[str, str, str],
                   rotor_positions: Tuple[int, int, int],
                   plugboard: Dict[str, str] = None) -> bool:
        """
        Test if a crib matches at current settings
        
        The key insight: if our settings are correct, encrypting the crib
        should produce the ciphertext
        """
        try:
            # Create a fresh Enigma with test settings
            enigma = EnigmaMachine(rotors, rotor_positions, plugboard=plugboard)
            encrypted_crib = enigma.encrypt(crib)
            return encrypted_crib == ciphertext[:len(crib)]
        except Exception as e:
            print(f"Error testing crib with settings {rotors}, {rotor_positions}: {e}")
            return False
    
    def break_enigma(self, ciphertext: str, crib: str, 
                    crib_position: int = 0,
                    max_positions: int = 17576,  # 26^3 possible rotor positions
                    plugboard: Dict[str, str] = None,
                    verbose: bool = True) -> Optional[Dict]:
        """
        Attempt to break Enigma encryption using a known crib
        
        Args:
            ciphertext: The encrypted message
            crib: Known plaintext fragment
            crib_position: Position where crib appears in plaintext
            max_positions: Maximum rotor positions to test (17576 = all positions)
            plugboard: Known plugboard settings (if any)
            verbose: Print progress
        
        Returns:
            Dictionary with found settings or None
        """
        crib = crib.upper().replace(' ', '')
        ciphertext = ciphertext.upper().replace(' ', '')
        
        if len(crib) > len(ciphertext):
            raise ValueError("Crib longer than ciphertext")
        
        # Extract the ciphertext portion we're trying to match
        crib_ciphertext = ciphertext[crib_position:crib_position + len(crib)]
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"BOMBE CRYPTANALYSIS STARTING")
            print(f"{'='*70}")
            print(f"Crib: '{crib}'")
            print(f"Crib position: {crib_position}")
            print(f"Target ciphertext section: '{crib_ciphertext}'")
            print(f"Testing {len(self.rotors_to_test)} rotor combinations")
            print(f"Max positions to test per rotor config: {max_positions:,}")
            print(f"{'='*70}\n")
        
        start_time = time.time()
        tests_performed = 0
        
        # Test each rotor combination
        for rotor_config in self.rotors_to_test:
            if verbose:
                print(f"Testing rotors: {rotor_config}")
            
            # Test rotor positions
            positions_tested = 0
            for pos in product(range(26), range(26), range(26)):
                if positions_tested >= max_positions:
                    if verbose:
                        print(f"  Reached max positions ({max_positions:,}) for this rotor config")
                    break
                
                tests_performed += 1
                positions_tested += 1
                
                if self._test_crib(crib_ciphertext, crib, rotor_config, pos, plugboard):
                    elapsed = time.time() - start_time
                    
                    if verbose:
                        print(f"\n{'='*70}")
                        print(f"✓ SOLUTION FOUND!")
                        print(f"{'='*70}")
                        print(f"Rotors: {rotor_config}")
                        print(f"Positions: {pos} ({chr(65+pos[0])}{chr(65+pos[1])}{chr(65+pos[2])})")
                        print(f"Tests performed: {tests_performed:,}")
                        print(f"Time elapsed: {elapsed:.2f} seconds")
                        print(f"{'='*70}\n")
                    
                    return {
                        'rotors': rotor_config,
                        'positions': pos,
                        'positions_letters': f"{chr(65+pos[0])}{chr(65+pos[1])}{chr(65+pos[2])}",
                        'tests_performed': tests_performed,
                        'time_seconds': elapsed,
                        'crib': crib,
                        'crib_position': crib_position
                    }
                
                if verbose and tests_performed % 1000 == 0:
                    print(f"  Tested {tests_performed:,} settings...", end='\r')
        
        if verbose:
            elapsed = time.time() - start_time
            print(f"\n✗ No solution found after {tests_performed:,} tests in {elapsed:.2f} seconds")
        
        return None
    
    def break_with_multiple_cribs(self, ciphertext: str, 
                                  cribs: List[Tuple[str, int]],
                                  max_positions: int = 17576,
                                  plugboard: Dict[str, str] = None,
                                  verbose: bool = True) -> Optional[Dict]:
        """
        Try breaking Enigma with multiple cribs
        
        Args:
            ciphertext: The encrypted message
            cribs: List of tuples (crib_text, crib_position)
            max_positions: Maximum positions to test per crib
            plugboard: Known plugboard settings
            verbose: Print progress
        
        Returns:
            Dictionary with found settings or None
        """
        print(f"\n{'='*70}")
        print(f"TESTING MULTIPLE CRIBS")
        print(f"{'='*70}")
        print(f"Number of cribs to try: {len(cribs)}\n")
        
        for i, (crib, position) in enumerate(cribs, 1):
            print(f"\nCRIB {i}/{len(cribs)}: '{crib}' at position {position}")
            print("-" * 70)
            
            result = self.break_enigma(
                ciphertext=ciphertext,
                crib=crib,
                crib_position=position,
                max_positions=max_positions,
                plugboard=plugboard,
                verbose=verbose
            )
            
            if result:
                result['successful_crib_number'] = i
                result['total_cribs_tested'] = i
                result['all_cribs'] = cribs
                return result
        
        # No solution found with any crib
        return {
            'success': False,
            'total_cribs_tested': len(cribs),
            'all_cribs': cribs,
            'message': 'No solution found with any of the provided cribs'
        }


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def decrypt_message(ciphertext: str, rotors: Tuple[str, str, str], 
                   positions: Tuple[int, int, int],
                   plugboard: Dict[str, str] = None) -> str:
    """
    Decrypt a message using found settings
    
    Args:
        ciphertext: The encrypted text
        rotors: Rotor configuration
        positions: Rotor positions
        plugboard: Plugboard settings
    
    Returns:
        Decrypted plaintext
    """
    enigma = EnigmaMachine(rotors, positions, plugboard=plugboard)
    return enigma.encrypt(ciphertext)  # Enigma is symmetric


def calculate_accuracy(decrypted_text: str, expected_text: str) -> float:
    """
    Calculate accuracy of decryption by comparing to expected text
    
    Args:
        decrypted_text: The decrypted message
        expected_text: The expected plaintext
    
    Returns:
        Accuracy percentage (0-100)
    """
    # Remove spaces and convert to uppercase for comparison
    decrypted = decrypted_text.upper().replace(' ', '')
    expected = expected_text.upper().replace(' ', '')
    
    min_len = min(len(decrypted), len(expected))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if decrypted[i] == expected[i])
    return (matches / min_len) * 100


def save_results_to_file(result: Dict, decrypted_text: str, 
                         ciphertext: str, original_plaintext: str,
                         accuracy: float, actual_settings: Dict = None,
                         output_file: str = "bombe_results.txt"):
    """
    Save Bombe results to a text file
    
    Args:
        result: Dictionary from break_enigma()
        decrypted_text: The decrypted message
        ciphertext: The original ciphertext
        original_plaintext: The original plaintext for comparison
        accuracy: Accuracy percentage
        actual_settings: The actual settings used for encryption (if known)
        output_file: Output filename
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("BOMBE CRYPTANALYSIS RESULTS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Timestamp: {timestamp}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("SETTINGS FOUND BY BOMBE\n")
        f.write("-"*80 + "\n")
        f.write(f"Rotor Configuration: {result['rotors']}\n")
        f.write(f"Rotor Positions (numeric): {result['positions']}\n")
        f.write(f"Rotor Positions (letters): {result['positions_letters']}\n\n")
        
        # Compare with actual settings if provided
        if actual_settings:
            f.write("-"*80 + "\n")
            f.write("ACTUAL ENCRYPTION SETTINGS (for comparison)\n")
            f.write("-"*80 + "\n")
            f.write(f"Rotor Configuration: {actual_settings['rotors']}\n")
            f.write(f"Rotor Positions (numeric): {actual_settings['positions']}\n")
            f.write(f"Rotor Positions (letters): {actual_settings['positions_letters']}\n\n")
            
            # Check if settings match
            rotors_match = result['rotors'] == actual_settings['rotors']
            positions_match = result['positions'] == actual_settings['positions']
            
            f.write("-"*80 + "\n")
            f.write("SETTINGS VERIFICATION\n")
            f.write("-"*80 + "\n")
            f.write(f"Rotors Match: {'✓ YES' if rotors_match else '✗ NO'}\n")
            f.write(f"Positions Match: {'✓ YES' if positions_match else '✗ NO'}\n")
            f.write(f"Settings Correctly Guessed: {'✓ YES' if (rotors_match and positions_match) else '✗ NO'}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("CRIB(S) USED\n")
        f.write("-"*80 + "\n")
        if 'all_cribs' in result:
            f.write(f"Total cribs tested: {result['total_cribs_tested']}\n")
            f.write(f"Successful crib number: {result['successful_crib_number']}\n\n")
            f.write("All cribs attempted:\n")
            for i, (crib, pos) in enumerate(result['all_cribs'], 1):
                marker = " ← SUCCESSFUL" if i == result['successful_crib_number'] else ""
                f.write(f"  {i}. '{crib}' at position {pos}{marker}\n")
        else:
            f.write(f"Crib: '{result['crib']}'\n")
            f.write(f"Crib position: {result['crib_position']}\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("PERFORMANCE METRICS\n")
        f.write("-"*80 + "\n")
        f.write(f"Time to solve: {result['time_seconds']:.2f} seconds\n")
        f.write(f"Settings tested: {result['tests_performed']:,}\n")
        f.write(f"Tests per second: {result['tests_performed']/result['time_seconds']:,.0f}\n")
        f.write(f"Search space explored: {result['tests_performed']/17576*100:.2f}% of single rotor config\n\n")
        
        f.write("-"*80 + "\n")
        f.write("DECRYPTION ACCURACY\n")
        f.write("-"*80 + "\n")
        f.write(f"Accuracy: {accuracy:.2f}%\n")
        f.write(f"Characters matched: {int(accuracy * len(original_plaintext) / 100):,} / {len(original_plaintext):,}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("CIPHERTEXT (first 200 chars)\n")
        f.write("-"*80 + "\n")
        f.write(f"{ciphertext[:200]}\n")
        if len(ciphertext) > 200:
            f.write(f"... ({len(ciphertext)} characters total)\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("DECRYPTED TEXT (first 500 chars)\n")
        f.write("-"*80 + "\n")
        f.write(f"{decrypted_text[:500]}\n")
        if len(decrypted_text) > 500:
            f.write(f"... ({len(decrypted_text)} characters total)\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("ORIGINAL PLAINTEXT (first 500 chars)\n")
        f.write("-"*80 + "\n")
        f.write(f"{original_plaintext[:500]}\n")
        if len(original_plaintext) > 500:
            f.write(f"... ({len(original_plaintext)} characters total)\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("FULL DECRYPTED TEXT\n")
        f.write("-"*80 + "\n")
        f.write(decrypted_text + "\n\n")
        
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"\n✓ Results saved to: {output_file}")




#### Main part of code to run 

In [7]:
def main():
    """
    Main function - Standalone Bombe decryption
    """
    print("\n" + "="*80)
    print("STANDALONE BOMBE CRYPTANALYSIS TOOL")
    print("="*80 + "\n")
    
    #==========================================================================
    # CONFIGURATION - Edit these paths and settings
    #==========================================================================
    encrypted_file = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Bee.txt"      # Your encrypted file
    plaintext_file = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Bee.txt"      # Original plaintext
    output_file = "bombe_results.txt"         # Where to save results
    
    # If you know the actual settings used for encryption, enter them here
    # for verification purposes (optional)
    actual_settings = {
        'rotors': ('I', 'II', 'III'),
        'positions': (5, 12, 18),  # Change to your actual positions
        'positions_letters': 'FMS'
    }
    # Set to None if you don't know the actual settings:
    # actual_settings = None
    #==========================================================================
    
    # Check if files exist
    if not os.path.exists(encrypted_file):
        print(f"ERROR: Could not find '{encrypted_file}'")
        print(f"Current directory: {os.getcwd()}\n")
        return
    
    if not os.path.exists(plaintext_file):
        print(f"WARNING: Could not find '{plaintext_file}'")
        print("Will proceed without accuracy calculation\n")
        plaintext_file = None
    
    # Read encrypted file
    print(f"Reading encrypted file: {encrypted_file}")
    with open(encrypted_file, 'r', encoding='utf-8') as f:
        ciphertext = f.read().strip()
    
    print("Cleaning ciphertext...")
    original_length = len(ciphertext)
    ciphertext = ''.join(c for c in ciphertext if c.isalpha()).upper()
    
    if original_length != len(ciphertext):
        print(f"  Removed {original_length - len(ciphertext)} non-letter characters")
    
    print(f"Ciphertext length: {len(ciphertext)} characters")
    print(f"First 100 characters: {ciphertext[:100]}")
    print()
    
    # Read plaintext if available
    original_plaintext = None
    if plaintext_file:
        print(f"Reading plaintext file: {plaintext_file}")
        with open(plaintext_file, 'r', encoding='utf-8') as f:
            original_plaintext = f.read().strip()
        original_plaintext = ''.join(c for c in original_plaintext if c.isalpha()).upper()
        print(f"Plaintext length: {len(original_plaintext)} characters\n")
    
    #==========================================================================
    # CRIBS - Configure based on known plaintext
    #==========================================================================
    cribs = [
        ("ACCORDING", 0),           # Bee Movie opening
        ("ACCORDINGTOALL", 0),      # Opening phrase
        ("AVIATION", 32),           # Later in first sentence
        ("BARRY", 0),               # Main character
        ("YELLOW", 0),              # About bees
        ("HONEY", 0),               # Common in Bee Movie
        ("BEE", 0),                 # Obviously
    ]
    #==========================================================================
    
    print("Cribs configured:")
    for i, (crib, pos) in enumerate(cribs, 1):
        print(f"  {i}. '{crib}' at position {pos}")
    print()
    
    # Initialize and run Bombe
    print("Initializing Bombe...")
    bombe = Bombe()
    
    print("Starting cryptanalysis...\n")
    result = bombe.break_with_multiple_cribs(
        ciphertext=ciphertext,
        cribs=cribs,
        max_positions=17576,  # Test all positions
        verbose=True
    )
    
    # Process results
    if result and result.get('success', True):
        print("\n" + "="*80)
        print("✓ SUCCESS! Settings found!")
        print("="*80 + "\n")
        
        # Decrypt full message
        print("Decrypting full message...")
        decrypted = decrypt_message(
            ciphertext=ciphertext,
            rotors=result['rotors'],
            positions=result['positions']
        )
        
        # Calculate accuracy
        accuracy = 0.0
        if original_plaintext:
            accuracy = calculate_accuracy(decrypted, original_plaintext)
            print(f"Accuracy: {accuracy:.2f}%\n")
        
        # Save results
        save_results_to_file(
            result=result,
            decrypted_text=decrypted,
            ciphertext=ciphertext,
            original_plaintext=original_plaintext or "",
            accuracy=accuracy,
            actual_settings=actual_settings,
            output_file=output_file
        )
        
        # Display summary
        print("\n" + "="*80)
        print("DECRYPTION COMPLETE!")
        print("="*80)
        print(f"\nSettings Found:")
        print(f"  Rotors: {result['rotors']}")
        print(f"  Positions: {result['positions_letters']} (numeric: {result['positions']})")
        
        if actual_settings:
            print(f"\nActual Settings:")
            print(f"  Rotors: {actual_settings['rotors']}")
            print(f"  Positions: {actual_settings['positions_letters']} (numeric: {actual_settings['positions']})")
            
            rotors_match = result['rotors'] == actual_settings['rotors']
            positions_match = result['positions'] == actual_settings['positions']
            
            print(f"\nVerification:")
            print(f"  Rotors match: {'✓ YES' if rotors_match else '✗ NO'}")
            print(f"  Positions match: {'✓ YES' if positions_match else '✗ NO'}")
            print(f"  Correctly guessed: {'✓ YES' if (rotors_match and positions_match) else '✗ NO'}")
        
        print(f"\nPerformance:")
        print(f"  Time to solve: {result['time_seconds']:.2f} seconds")
        print(f"  Tests performed: {result['tests_performed']:,}")
        print(f"  Accuracy: {accuracy:.2f}%")
        print(f"\nResults saved to: {output_file}")
        print("="*80 + "\n")
        
    else:
        print("\n" + "="*80)
        print("✗ FAILED TO DECRYPT")
        print("="*80)
        print("No solution found with the provided cribs.")
        print("\nTroubleshooting:")
        print("1. Verify the encrypted file is correct")
        print("2. Check that cribs match the actual plaintext")
        print("3. Ensure file was encrypted with rotors I, II, III")
        print("4. Try adding more cribs based on known content")
        print("="*80 + "\n")


if __name__ == "__main__":
    main()



STANDALONE BOMBE CRYPTANALYSIS TOOL

Reading encrypted file: C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Bee.txt
Cleaning ciphertext...
Ciphertext length: 37231 characters
First 100 characters: FYVZUEVCEQDWVHMYEMBHTPDDUDYUFLMJDMGLSSAPPTDKMCDWRWRAOWNCYHRESEIJWHXSYTHBQLMYVOXDESZSHQAJYAKHTIZYJQML

Reading plaintext file: C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Bee.txt
Plaintext length: 37231 characters

Cribs configured:
  1. 'ACCORDING' at position 0
  2. 'ACCORDINGTOALL' at position 0
  3. 'AVIATION' at position 32
  4. 'BARRY' at position 0
  5. 'YELLOW' at position 0
  6. 'HONEY' at position 0
  7. 'BEE' at position 0

Initializing Bombe...
Starting cryptanalysis...


TESTING MULTIPLE CRIBS
Number of cribs to try: 7


CRIB 1/7: 'ACCORDING' at position 0
----------------------------------------------------------------------

BOMBE CRYPTANALYSIS STARTING
Crib: 'ACCORDING'
Crib position: 0
Target ciphertext section: '